# Experimental setup

Set up for the experiment. This will load in the required packages, load in the data set, and load in the model. Then the experiment is run on the data set, saving the model weights and the results so they can be reloaded later. Below are the returned metrics.

## Metrics

- Overall performance in learned tasks: Usually average accuracy and average incremental accuracy, and confusion matrix
- Memory stability of old classes: uses forgetting measure (average forgetting of old tasks) and backward transfer (average influence of learning the k-th task on all old tasks)
- Learning plasticity of new classes: intransience measure (inability to learn a new task) and forward transfer (average influence of all old tasks on the current task)
- Resource (storage) and computational overheads (big O complexity), RAM usage.
- Discrepancy of task distribution

In [ ]:
#TODO:
# Alter to use separate training functions for each training group and separate test functions out from the training function

## Imports

In [4]:
# Load packages
import torch
from torch import nn
from torch.utils.data import DataLoader

import torchvision
from torchvision import datasets
from torchvision.transforms import transforms


import numpy as np
import matplotlib.pyplot as plt

import os
import time

from collections import defaultdict

print(torch.__version__)
print(torch.cuda.is_available())
torch.set_default_device('cuda')

2.10.0+cu128
True


In [ ]:
# Parameters

LOAD_FROM_FILE = False
FILE_LOAD_PATH = os.path.join(os.getcwd(), 'models', 'model_weights.pth')

TEST_INTERVAL = 'class' # Can be 'class', 'batch', or None. 
TEST_EVERY_N = 100 # Number of batches between tests, only used if TEST_INTERVAL='batch'


In [6]:
# Load dataset and model
dataset_path = os.path.join(os.getcwd(), 'datasets')

download = not (os.path.exists(dataset_path))

print('Downloading dataset:',download)

print('Dataset path:', dataset_path)

# Define the desired size for the input tensors
desired_size = (224, 224)

# Create a transformation to resize the input tensors
resize_transform = transforms.Compose([
    transforms.Resize(desired_size),
    transforms.ToTensor()
])

dataset_train = torchvision.datasets.Imagenette(root = dataset_path, 
                                          split = 'train', 
                                          download = download, transform = resize_transform
                                          )

dataset_test = torchvision.datasets.Imagenette(root = dataset_path, 
                                          split = 'val', 
                                          download = download, transform = resize_transform
                                          )

def group_indices_by_class(dataset):
    class_to_indices = defaultdict(list)
    for idx in range(len(dataset)):
        _, y = dataset[idx]
        class_to_indices[int(y)].append(idx)
    return class_to_indices

class ContinualClassStream(torch.utils.data.IterableDataset):
    def __init__(self, dataset, class_order, class_to_indices):
        self.dataset = dataset
        self.class_order = class_order
        self.class_to_indices = class_to_indices

    def __iter__(self):
        for cls in self.class_order:
            for idx in self.class_to_indices[cls]:
                yield self.dataset[idx]

class_to_indices_train = group_indices_by_class(dataset_train)
class_order_train = sorted(class_to_indices_train.keys())  # or any order you want

train_stream = ContinualClassStream(dataset_train, class_order_train, class_to_indices_train)

train_dataloader = DataLoader(
    train_stream,
    batch_size=1,
    num_workers=0  # IMPORTANT for online streams
)

class_to_indices_test = group_indices_by_class(dataset_test)
class_order_test = sorted(class_to_indices_test.keys())  # or any order you want

test_stream = ContinualClassStream(dataset_test, class_order_test, class_to_indices_test)
test_dataloader = DataLoader(
    test_stream,
    batch_size=1,
    num_workers=0  # IMPORTANT for online streams
)

model = torchvision.models.convnext_tiny()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
print(f"Using {device} device")
print(model)


Dataset path: c:\Users\naido\Documents\ChalmersCourses\Thesis\continous-MoE\datasets
Using cuda device
ConvNeXt(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
      (1): LayerNorm2d((96,), eps=1e-06, elementwise_affine=True)
    )
    (1): Sequential(
      (0): CNBlock(
        (block): Sequential(
          (0): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
          (1): Permute()
          (2): LayerNorm((96,), eps=1e-06, elementwise_affine=True)
          (3): Linear(in_features=96, out_features=384, bias=True)
          (4): GELU(approximate='none')
          (5): Linear(in_features=384, out_features=96, bias=True)
          (6): Permute()
        )
        (stochastic_depth): StochasticDepth(p=0.0, mode=row)
      )
      (1): CNBlock(
        (block): Sequential(
          (0): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
          (1): Permute()

In [7]:
# Create training metaparameters

# TODO: Probably has minibatches within a class, and each class is separated completely. Either test between each minibatch or between each added class
# Class incremental online learning: Should have the data slowly add in new classes. 
# Then train on the new classes. 
# The old classes should maybe have a few new examples over time? Or just purely the new classes?

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

epochs = 1

In [ ]:
# Training fcns

def train(dataloader, model, loss_fn, optimizer, test_dataloader=None, test_fn=None, 
          test_interval='class', test_every_n=100, class_order=None):
    """
    Train on continual learning stream with per-class tracking and optional intermediate testing.
    
    Args:
        dataloader: DataLoader with ContinualClassStream
        model: Model to train
        loss_fn: Loss function
        optimizer: Optimizer
        test_dataloader: Optional DataLoader for testing between training phases
        test_fn: Test function to call (should be the test function)
        test_interval: 'class' (test after each class), 'batch' (test every N batches), or None (no intermediate testing)
        test_every_n: Number of batches between tests (only used if test_interval='batch')
        class_order: List of class labels in order (for per-class metrics)
    
    Returns:
        Dictionary with training metrics per class
    """
    model.train()
    batch_count = 0
    current_class = None
    class_batch_counts = defaultdict(int)
    class_losses = defaultdict(float)
    training_metrics = {}
    
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        y_class = int(y[0].item())  # Get class label from batch
        
        # Track when we transition to a new class
        if current_class != y_class:
            if current_class is not None:
                avg_loss = class_losses[current_class] / class_batch_counts[current_class]
                training_metrics[current_class] = {
                    'samples': class_batch_counts[current_class],
                    'avg_loss': avg_loss
                }
                print(f"  Class {current_class} - Processed {class_batch_counts[current_class]} samples, Avg loss: {avg_loss:>7f}")
                
                # Test after each class if requested
                if test_interval == 'class' and test_dataloader is not None and test_fn is not None:
                    print(f"  Testing after Class {current_class}:")
                    test_fn(test_dataloader, model, loss_fn, class_order=class_order)
                    model.train()  # Return to training mode
            
            current_class = y_class
            print(f"Starting training on Class {current_class}")

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        class_batch_counts[current_class] += 1
        class_losses[current_class] += loss.item()
        batch_count += 1

        if batch % 100 == 0 and batch > 0:
            print(f"  Batch {batch}: loss: {loss.item():>7f}")
            
            # Test every N batches if requested
            if test_interval == 'batch' and batch % test_every_n == 0 and test_dataloader is not None and test_fn is not None:
                print(f"  Testing at Batch {batch}:")
                test_fn(test_dataloader, model, loss_fn, class_order=class_order)
                model.train()  # Return to training mode
    
    # Print final class stats
    if current_class is not None:
        avg_loss = class_losses[current_class] / class_batch_counts[current_class]
        training_metrics[current_class] = {
            'samples': class_batch_counts[current_class],
            'avg_loss': avg_loss
        }
        print(f"  Class {current_class} - Processed {class_batch_counts[current_class]} samples, Avg loss: {avg_loss:>7f}")
        
        # Final test after last class if requested
        if test_interval == 'class' and test_dataloader is not None and test_fn is not None:
            print(f"  Testing after Class {current_class}:")
            test_fn(test_dataloader, model, loss_fn, class_order=class_order)
            model.train()  # Return to training mode
    
    print(f"Training complete. Total batches: {batch_count}\n")
    return training_metrics


def test(dataloader, model, loss_fn, class_order=None):
    """
    Test on continual learning stream with per-class and overall metrics.
    
    Args:
        dataloader: DataLoader with ContinualClassStream
        model: Model to evaluate
        loss_fn: Loss function
        class_order: List of class labels in order (for per-class metrics)
    
    Returns:
        Dictionary with test metrics per class and overall metrics
    """
    model.eval()
    
    test_loss = 0.0
    correct = 0
    total = 0
    
    current_class = None
    class_correct = defaultdict(int)
    class_total = defaultdict(int)
    class_losses = defaultdict(float)
    class_batch_counts = defaultdict(int)
    test_metrics = {}
    
    with torch.no_grad():
        for batch, (X, y) in enumerate(dataloader):
            X, y = X.to(device), y.to(device)
            y_class = int(y[0].item())  # Get class label from batch
            
            # Track when we transition to a new class
            if current_class != y_class:
                if current_class is not None:
                    class_acc = (100 * class_correct[current_class] / class_total[current_class])
                    class_avg_loss = class_losses[current_class] / class_batch_counts[current_class]
                    test_metrics[current_class] = {
                        'accuracy': class_acc,
                        'avg_loss': class_avg_loss,
                        'total_samples': class_total[current_class]
                    }
                    print(f"  Class {current_class} - Accuracy: {class_acc:>0.1f}%, Avg loss: {class_avg_loss:>8f}")
                current_class = y_class
                print(f"Testing on Class {current_class}")
            
            pred = model(X)
            loss = loss_fn(pred, y)
            
            test_loss += loss.item()
            total += y.size(0)
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
            
            # Per-class tracking
            y_class_label = int(y[0].item())
            class_total[y_class_label] += 1
            class_losses[y_class_label] += loss.item()
            class_batch_counts[y_class_label] += 1
            class_correct[y_class_label] += (pred.argmax(1) == y).type(torch.float).sum().item()
    
    # Print final class stats
    if current_class is not None:
        class_acc = (100 * class_correct[current_class] / class_total[current_class])
        class_avg_loss = class_losses[current_class] / class_batch_counts[current_class]
        test_metrics[current_class] = {
            'accuracy': class_acc,
            'avg_loss': class_avg_loss,
            'total_samples': class_total[current_class]
        }
        print(f"  Class {current_class} - Accuracy: {class_acc:>0.1f}%, Avg loss: {class_avg_loss:>8f}")
    
    # Overall metrics
    avg_loss = test_loss / (batch + 1) if batch >= 0 else 0
    overall_accuracy = 100 * correct / total if total > 0 else 0
    test_metrics['overall'] = {
        'accuracy': overall_accuracy,
        'avg_loss': avg_loss,
        'total_samples': total
    }
    print(f"\nTest Summary:")
    print(f"  Overall Accuracy: {overall_accuracy:>0.1f}%")
    print(f"  Overall Avg Loss: {avg_loss:>8f}\n")
    
    return test_metrics


In [ ]:
# Training

# Option 1: Test after each class
if TEST_INTERVAL == 'class':
    train_metrics = train(train_dataloader, model, loss_fn, optimizer, 
                        test_dataloader=test_dataloader, test_fn=test, 
                        test_interval='class')

# Option 2: Test every N batches (e.g., every 100 batches)

elif TEST_INTERVAL == 'batch':
    train_metrics = train(train_dataloader, model, loss_fn, optimizer, 
                        test_dataloader=test_dataloader, test_fn=test, 
                        test_interval='batch', test_every_n=100)

# Option 3: No intermediate testing

else:
    for t in range(epochs):
        print(f"Epoch {t+1}\n-------------------------------")
        train(train_dataloader, model, loss_fn, optimizer)
        test(test_dataloader, model, loss_fn)

##################

print("Done!")

timestr = time.strftime("%Y%m%d-%H%M%S")

torch.save(model.state_dict(), os.path.join(os.getcwd(), 'models', timestr + 'model_weights.pth'))
print("Saved PyTorch Model State to " + timestr + "model_weights.pth")


Epoch 1
-------------------------------
Starting training on Class 0
  Batch 100: loss: 0.005120
  Batch 200: loss: 0.002918
  Batch 300: loss: 0.002002
  Batch 400: loss: 0.001521
  Batch 500: loss: 0.001559
  Batch 600: loss: 0.001105
  Batch 700: loss: 0.000927
  Batch 800: loss: 0.000776
  Batch 900: loss: 0.000968
  Class 0 - Processed 963 samples, Avg loss: 0.016711
Starting training on Class 1
  Batch 1000: loss: 0.014881
  Batch 1100: loss: 0.004302
  Batch 1200: loss: 0.002804
  Batch 1300: loss: 0.001822
  Batch 1400: loss: 0.001459
  Batch 1500: loss: 0.001217
  Batch 1600: loss: 0.001049
  Batch 1700: loss: 0.000885
  Batch 1800: loss: 0.000796
  Batch 1900: loss: 0.000736
  Class 1 - Processed 955 samples, Avg loss: 0.028882
Starting training on Class 2
  Batch 2000: loss: 0.007696
  Batch 2100: loss: 0.003513
  Batch 2200: loss: 0.002311
  Batch 2300: loss: 0.001857
  Batch 2400: loss: 0.001386
  Batch 2500: loss: 0.001227
  Batch 2600: loss: 0.001002
  Batch 2700: loss: 

In [ ]:
# Metrics and plots

# TODO: 
# Add in confusion matrix for the classes
# Overall performance in learned tasks: Usually average accuracy and average incremental accuracy over all tasks
# Memory stability of old classes: uses forgetting measure (average forgetting of old tasks) and backward transfer (average influence of learning the k-th task on all old tasks)
# Learning plasticity of new classes: intransience measure (inability to learn a new task) and forward transfer (average influence of all old tasks on the current task)
# Resource (storage) and computational overheads (big O complexity), RAM usage.
# Discrepancy of task distribution